# The Spark ecosystem: where Spark runs, Delta Lake, and when to actually use it

_Apache Spark_

**How Spark runs in the real world — local, Databricks, EMR, Dataproc — plus Delta Lake for a reliable data lake, and the judgment of when a simpler tool wins.**

Spark is an engine, but in the real world it always runs inside something, reads from
       somewhere, and competes with other tools. Understanding the ecosystem is three layers.

       Layer 1 &mdash; where the engine runs. At the bottom is a cluster manager that owns the
       machines and hands Spark workers their share:

---

This notebook is a practice scaffold for the **The Spark ecosystem: where Spark runs, Delta Lake, and when to actually use it** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PySpark + delta

In [ ]:
# Colab: !pip install pyspark delta-spark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip, DeltaTable

# --- Build a LOCAL-MODE Spark session WITH Delta wired in -----------------
# (On Databricks/EMR/Dataproc you skip this — 'spark' is created for you with
#  Delta already configured. See the platform launch notes at the bottom.)
builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("delta-demo")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

path = "/tmp/customers_delta"

# --- 1. WRITE a DataFrame as a Delta table (version 0) --------------------
df0 = spark.createDataFrame(
    [(1, "Ada", "NY"), (2, "Lin", "CA")],
    ["id", "name", "state"],
)
df0.write.format("delta").mode("overwrite").save(path)   # bare Parquet + _delta_log/

# --- 2. READ it back ------------------------------------------------------
spark.read.format("delta").load(path).show()
# id | name | state
#  1 | Ada  | NY
#  2 | Lin  | CA

# --- 3. UPSERT via MERGE: update Lin (moved to WA), insert new Ravi -------
updates = spark.createDataFrame(
    [(2, "Lin", "WA"), (3, "Ravi", "TX")],
    ["id", "name", "state"],
)
delta_tbl = DeltaTable.forPath(spark, path)
(
    delta_tbl.alias("t")
    .merge(updates.alias("u"), "t.id = u.id")
    .whenMatchedUpdateAll()        # id 2 exists -> UPDATE in place
    .whenNotMatchedInsertAll()     # id 3 is new -> INSERT
    .execute()                     # one atomic commit -> version 1
)
spark.read.format("delta").load(path).orderBy("id").show()
# id | name | state
#  1 | Ada  | NY
#  2 | Lin  | WA   <- updated
#  3 | Ravi | TX   <- inserted

# --- 4. TIME TRAVEL: read the table AS OF version 0 -----------------------
old = spark.read.format("delta").option("versionAsOf", 0).load(path)
old.orderBy("id").show()
# id | name | state   (the ORIGINAL two rows — Lin still in CA, no Ravi)
#  1 | Ada  | NY
#  2 | Lin  | CA

# --- PLATFORM LAUNCH NOTES (same code, different "where it runs") ---------
# Local / Colab : pip install pyspark delta-spark  (the session built above)
# Databricks    : Delta is the DEFAULT format; 'spark' already exists.
#                 Free to learn on Databricks Community Edition.
# AWS EMR       : aws emr create-cluster --applications Name=Spark ;
#                 spark-submit --packages io.delta:delta-spark_2.12:<ver> job.py
# GCP Dataproc  : gcloud dataproc clusters create my-cl --region=us-central1 ;
#                 gcloud dataproc jobs submit pyspark job.py --cluster=my-cl
# Cluster manager underneath: YARN (EMR/Hadoop), Kubernetes (modern), or Standalone.

## Visualize the data & results

_For one group-by aggregation over a mid-size (~50 GB) Parquet dataset, how long does each tool take — and at which data size does each tool actually win?_

In [ ]:
import numpy as np
import pandas as pd

# A reproducible COST MODEL (not a real benchmark) that ILLUSTRATES where each
# tool wins for a group-by aggregation, by data size. Numbers are plausible.

# --- per-tool cost model: runtime(seconds) = fixed_overhead + per_GB * size ---
# fixed_overhead = startup/planning/shuffle; per_gb = throughput;
# ram_cap = data size (GB) above which a single-node tool runs out of memory.
tools = {
    #                fixed   per_gb   ram_cap(GB)
    "pandas":              (0.2,  3.00,   8),    # in-memory only, smallest cap
    "DuckDB":              (0.3,  0.70, 120),    # single-node, vectorized, fast
    "Polars":              (0.3,  0.55, 120),    # single-node, lazy, fast
    "Spark (10-node)":     (12.0, 0.20, 1e9),    # heavy startup, scales out
    "warehouse (BigQuery)":(2.0,  0.14, 1e9),    # managed, columnar, scales out
}

def runtime(name, size_gb):
    fixed, per_gb, cap = tools[name]
    if size_gb > cap:
        return np.inf                 # out-of-memory: cannot finish
    return fixed + per_gb * size_gb

sizes = {"small": 1, "mid": 50, "large": 2000}   # GB

# Print the full size x tool grid so the "who wins where" story is explicit.
print(f'{"tool":22s} {"small(1GB)":>11s} {"mid(50GB)":>11s} {"large(2TB)":>12s}')
for name in tools:
    row = []
    for s in sizes.values():
        t = runtime(name, s)
        row.append("OOM" if np.isinf(t) else f"{t:8.1f}s")
    print(f'{name:22s} {row[0]:>11s} {row[1]:>11s} {row[2]:>12s}')

# The plotted bars = the MID (50 GB) column (pandas OOM shown as a tall bar).
mid = {n: runtime(n, sizes["mid"]) for n in tools}
plotted = {n: (999 if np.isinf(t) else round(t)) for n, t in mid.items()}
print("\nplotted (50 GB, OOM->999):", plotted)
# tool                    small(1GB)   mid(50GB)   large(2TB)
# pandas                       3.2s        OOM          OOM
# DuckDB                       1.0s       35.3s          OOM
# Polars                       0.9s       27.8s          OOM
# Spark (10-node)             12.2s       22.0s        412.0s
# warehouse (BigQuery)         2.1s        9.0s        282.0s
# plotted (50 GB, OOM->999): {'pandas': 999, 'DuckDB': 35, 'Polars': 28,
#                             'Spark (10-node)': 22, 'warehouse (BigQuery)': 9}

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate wants to spin up a 12-node Spark cluster to run a nightly GROUP BY aggregation over a 40 GB Parquet dataset. The result is a small summary table. Is Spark the right call, and what would you suggest checking first?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Size the data honestly: 40 GB of Parquet, and the job is a single group-by aggregation producing a small output. — _40 GB comfortably fits the memory or fast local disk of one large cloud machine, so it does not require distribution._
- Try a single-node engine first: DuckDB (SELECT region, SUM(amount) ... GROUP BY region straight over the Parquet) or Polars lazy scan. — _Below ~100 GB these are usually faster than Spark because they skip JVM startup, job planning, and network shuffles._
- Reserve Spark for when one machine genuinely cannot fit or finish the job, or when the team already runs everything on a Spark platform. — _A cluster adds latency, cost, and operational overhead that small-to-mid data does not repay._

**Answer:** Probably not. 40 GB is squarely in single-node territory: a DuckDB or Polars query on one fat machine will likely run the aggregation faster than a 12-node Spark cluster, with no cluster to provision, pay for, or babysit. Spark's per-job overhead (JVM startup, planning, shuffle) only pays off when data truly exceeds one machine (roughly >100 GB&ndash;TB scale) or when the team's whole platform is already Spark. Check the single-node path first; reach for the cluster only if it genuinely cannot fit or finish.

</details>

**Problem 2.** Your data lake is a directory of raw Parquet files. A nightly load occasionally fails halfway, leaving readers seeing a half-written table, and last week a column's type silently changed and corrupted a downstream report. Name the technology that fixes this and the two specific guarantees that solve these exact problems.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the symptoms: partial writes visible to readers, and an unnoticed schema change. These are the hallmarks of a "data swamp" — raw Parquet with no transactions. — _Bare Parquet has no atomic commit and no schema check, so a crashed write and a drifting type both slip through._
- Convert the table to Delta Lake (write format("delta") instead of bare Parquet). — _Delta adds an ordered transaction log on top of the same Parquet files._
- Map each symptom to a guarantee: atomic commits for the half-write, schema enforcement for the type drift. — _One transaction log gives both — a write is all-or-nothing, and a mismatched schema is rejected._

**Answer:** Use Delta Lake. The two guarantees: ACID transactions / atomic commits &mdash; a write becomes visible only as a single all-or-nothing commit, so a job that dies halfway leaves the previous version intact and readers never see a half-written table; and schema enforcement &mdash; a write whose column types do not match the table is rejected rather than silently absorbed, so the corrupting type change would have been blocked. As a bonus, Delta's time travel would have let you read the pre-corruption version to recover.

</details>

**Problem 3.** You loaded a Delta table yesterday (version 3). This morning a bad daily feed overwrote good rows (now version 4). The data is gone from the current table. How do you recover it, and what feature makes this possible?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the table as of the known-good version: spark.read.format("delta").option("versionAsOf", 3).load(path). — _Delta's time travel reconstructs the exact file set described by version 3's commit — nothing was physically deleted._
- Write that older snapshot back as a new commit (or use RESTORE TO VERSION AS OF 3), creating version 5 with the good data. — _Restoring is itself just another atomic commit, so it is safe and also undoable._
- Going forward, add schema enforcement / validation so a bad feed is rejected rather than committed. — _Prevention beats recovery; Delta's schema enforcement can stop the next bad load._

**Answer:** Use time travel: read the table option("versionAsOf", 3) to get yesterday's good snapshot, then write it back (or RESTORE TO VERSION AS OF 3), which lands as a new version with the recovered rows. This works because Delta records every version in its transaction log and does not physically delete old data files until you explicitly vacuum &mdash; so any past version stays queryable and restorable. A raw Parquet lake, with no history, would have lost the rows for good.

</details>